In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q vllm fastapi uvicorn pyngrok mlflow sentence-transformers

In [ ]:
# 2. Setup ngrok auth
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN") # <--- THAY TOKEN CỦA BẠN VÀO ĐÂY

In [ ]:
import subprocess, threading, time
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn

# 3. Khởi động vLLM Server (Model Chat)
def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85"
    ])

print("Đang khởi động vLLM Server (mất khoảng 1-2 phút)...")
threading.Thread(target=run_vllm, daemon=True).start()
time.sleep(60) # Chờ model load xong

tunnel_vllm = ngrok.connect(8001, "http")
print(f"✅ vLLM URL (copy cái này): {tunnel_vllm.public_url}")

In [ ]:
# 4. Khởi động Embedding Server (Model Vector)
app = FastAPI()
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

@app.post("/embed")
def embed(data: dict):
    texts = data["texts"]
    embeddings = model.encode(texts).tolist()
    return {"embeddings": embeddings}

def run_embed():
    uvicorn.run(app, host="0.0.0.0", port=8002)

print("Đang khởi động Embedding Server...")
threading.Thread(target=run_embed, daemon=True).start()
time.sleep(10) # Chờ model embedding load

tunnel_embed = ngrok.connect(8002, "http")
print(f"✅ Embedding URL (copy cái này): {tunnel_embed.public_url}")